# 6. Testing the things that must not fail

Every bug in this project so far was found the same way: by me poking at
the running system and reading output. That works, and it does not scale,
and it only ever checks the path I happened to walk.

This notebook is about the second kind of testing - not "does this
function return the right value" but "can this system ever do the thing
it must never do".


## The bugs that motivated it

Five in one evening. Worth listing, because they have a shape:

| bug | where it lived |
|---|---|
| `fee` matched `feet` | inside one function |
| greetings escalated to a human | inside one prompt |
| the demo link failed grounding | between the prompt and the KB |
| an unnamed topic produced silence | between the classifier and the KB |
| dedup suppressed a needed reply | between two rules that were each correct |

The first two are ordinary unit-test material. The last three are not -
every component was behaving as designed and the *combination* was
wrong.

That last one is worth sitting with.


### Two correct rules, one broken system

**Rule A.** A scope block leaves autopilot running. A single pricing
question shouldn't end an automated conversation.

**Rule B.** Don't send a canned holding reply identical to the previous
outbound message. Repeating yourself reads as broken.

Both sensible. Both were the right call when written. Then rule A changed
- scope blocks started disabling autopilot - and rule B silently became
harmful, because a path that can no longer repeat doesn't need dedup, and
suppressing there means the prospect gets nothing.

Nothing in the code connected them. Let me show the failure.


In [ ]:
conversation = []          # outbound messages, oldest last

HOLDING = 'Good question - I would rather get you exact numbers than guess.'

def send_holding(text, dedup=True):
    if dedup and conversation and conversation[-1] == text:
        return 'suppressed as duplicate'
    conversation.append(text)
    return 'sent'

# Earlier in the conversation, a pricing question was blocked.
print(send_holding(HOLDING))

# Operator answers, re-enables autopilot. Prospect asks about price again.
print(send_holding(HOLDING))          # <- the bug
print('\nprospect received:', len(conversation), 'message(s) for 2 questions')


The second pricing question got silence. From the prospect's side: they
asked about price, and nothing came back.

No unit test would catch this. `send_holding` is doing exactly what it
was written to do. The bug is that its correctness depended on a fact
about a different rule, and that fact changed.

So the test has to assert the *property*, not the function:


In [ ]:
def test_a_blocked_message_always_produces_a_reply():
    conversation.clear()
    send_holding(HOLDING, dedup=False)
    send_holding(HOLDING, dedup=False)
    assert len(conversation) == 2, 'a blocked message produced silence'
    return 'pass'

print(test_a_blocked_message_always_produces_a_reply())


## Fakes, not mocks

Most of these properties involve the database. Suppression lives in a
table; the question "does an opt-out survive prospect deletion?" is a
question about stored state.

The reflex is to mock. Here is why that fails.


In [ ]:
from unittest.mock import MagicMock

db = MagicMock()

def delete_prospect_buggy(prospect_id):
    db.table('prospects').delete().eq('id', prospect_id).execute()
    # BUG: cascade also wipes the suppression
    db.table('suppressions').delete().eq('prospect_id', prospect_id).execute()

delete_prospect_buggy('p1')

# The mock-flavoured assertion:
print('delete was called:', db.table.called)
print('=> test passes, and the opt-out is gone')


The mock confirms a function was called. It cannot answer whether the
suppression survived, because it never stored anything.

A fake does. It is more code up front and it earns that back the first
time a test asks a question about state.


In [ ]:
class FakeDB:
    def __init__(self):
        self.tables = {}

    def insert(self, table, row):
        self.tables.setdefault(table, []).append(row)

    def delete(self, table, **match):
        rows = self.tables.setdefault(table, [])
        self.tables[table] = [
            r for r in rows if not all(r.get(k) == v for k, v in match.items())
        ]

    def exists(self, table, **match):
        return any(all(r.get(k) == v for k, v in match.items())
                   for r in self.tables.get(table, []))

db = FakeDB()
db.insert('prospects', {'id': 'p1', 'phone': '+14165551234'})
db.insert('suppressions', {'phone': '+14165551234', 'prospect_id': 'p1'})

def delete_prospect_buggy(pid):
    db.delete('prospects', id=pid)
    db.delete('suppressions', prospect_id=pid)      # the bug

delete_prospect_buggy('p1')
print('prospect gone      :', not db.exists('prospects', id='p1'))
print('opt-out survived   :', db.exists('suppressions', phone='+14165551234'))
print('=> test FAILS, correctly')


Same test, real answer. The fake stores rows, so the assertion is about
the world rather than about a call.

This matters more than it looks. The failing case is not hypothetical:
deleting a prospect cascades to their messages, and if the opt-out lived
on the prospect row rather than in its own table, re-importing the same
lead CSV a month later would message someone who said stop.

## Testing an ordering

Some guarantees are about *sequence*, and sequence is invisible from
outside. The STOP check has to run before the triage agent - not because
the outcome differs today, but because a model deciding whether "STOP"
means stop is the wrong shape for a legal obligation.

Both of these pass an outcome test:


In [ ]:
STOP_KEYWORDS = {'stop', 'unsubscribe', 'quit'}

def handle_correct(body):
    if body.lower() in STOP_KEYWORDS:      # keyword first
        return 'opted_out'
    return classify_with_model(body)

def handle_risky(body):
    intent = classify_with_model(body)     # model first
    if intent == 'opt_out' or body.lower() in STOP_KEYWORDS:
        return 'opted_out'
    return intent

def classify_with_model(body):
    return 'opt_out' if 'stop' in body.lower() else 'question'

for fn in (handle_correct, handle_risky):
    print(f'{fn.__name__:16} STOP -> {fn("STOP")}')


Identical. But `handle_risky` sends every opt-out through a model first,
so a classifier outage or a bad sample becomes a compliance failure.

The only way to pin this is to assert on the source. It is unusual and
brittle to renames, and it is the sole option when the ordering *is* the
guarantee.


In [ ]:
# Reading the source as text rather than via inspect.getsource, so this
# runs the same way everywhere. The real suite uses inspect on imported
# modules, where it works fine.
CORRECT = '''
def handle(body):
    if body.lower() in STOP_KEYWORDS:
        return "opted_out"
    return classify_with_model(body)
'''

RISKY = '''
def handle(body):
    intent = classify_with_model(body)
    if intent == "opt_out" or body.lower() in STOP_KEYWORDS:
        return "opted_out"
    return intent
'''

def test_stop_check_precedes_the_model(source):
    keyword_at = source.index('STOP_KEYWORDS')
    model_at = source.index('classify_with_model')
    return 'pass' if keyword_at < model_at else 'FAIL - model runs first'

for name, source in [('correct', CORRECT), ('risky', RISKY)]:
    print(f'{name:9} {test_stop_check_precedes_the_model(source)}')


Four tests in the real suite work this way: that `is_suppressed` appears
*inside* `send_sms` rather than at each call site, that the STOP check
precedes `classify()`, that `dedup=False` is on the scope-block path.

I am not fully comfortable with them. They break on a rename and they
couple tests to implementation. But each one pins a property that cannot
be observed from outside, and every one of them corresponds to a mistake
that was actually made.

## Verifying the tests

A test that has never failed has not been tested. The suite was checked
by deliberately breaking the code twice:

```
# disable the suppression check in send_sms
-    if is_suppressed(to_phone):
+    if False and is_suppressed(to_phone):

=> FAILED test_send_to_suppressed_number_raises

# remove the dedup exemption - the actual bug from that evening
-    _send_holding_reply(..., tr, dedup=False)
+    _send_holding_reply(..., tr)

=> FAILED test_scope_blocks_do_not_dedup
   'the scope-block holding reply is deduping again - if the same
    line was the last outbound, the prospect gets silence'
```

The second one is the point of the whole exercise. That bug took a live
transcript and twenty minutes to find. It now fails in two seconds.

## What this became

| here | in the repo |
|---|---|
| `FakeDB` | `tests/fakes.py` - `FakeSupabase`, `FakeTwilio` |
| the property tests | `tests/test_compliance.py`, 20 of them |
| `inspect.getsource` checks | 4 tests pinning placement and ordering |

The real fake implements the query shapes this codebase uses and raises
on anything unfamiliar, rather than quietly returning nothing. A fake
that answers wrong is worse than one that fails loudly.

## What I would take from this

**Unit tests catch bugs inside components. Most of mine were between
them.** Every guardrail was behaving as designed on the evening it
produced silence.

**Assert properties, not functions.** "A blocked message always produces
a reply" survives a refactor. "`send_holding` returns X" does not, and
does not say anything about the obligation.

**Use a fake when the question is about state.** A mock answers "was it
called". Half these tests need "what is true now".

**Break the code to check the test.** Two deliberate regressions took a
minute and turned twenty assertions from decoration into evidence.
